# HemoMesh Colab GEM-GCN Baseline

This notebook stages the pretrained Suk et al. GEM-GCN baseline reproduction in a Colab GPU runtime. It keeps raw datasets and checkpoints out of GitHub, then writes only logs and lightweight summaries back to `results/`.

Run this notebook with **Runtime > Change runtime type > GPU**.

## 1. Check Runtime

In [1]:
!nvidia-smi
!python --version

Wed Jul  8 03:42:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone HemoMesh And Upstream Baseline Code

The HemoMesh repository is public, so Colab can clone it directly.

In [2]:
import os
import subprocess
from pathlib import Path

PROJECT_REPO = "https://github.com/Lawson-Darrow/HemoMesh.git"
UPSTREAM_REPO = "https://github.com/sukjulian/coronary-mesh-convolution.git"


def run(command):
    subprocess.run(command, check=True)


run(["rm", "-rf", "/content/HemoMesh"])
run(["git", "clone", PROJECT_REPO, "/content/HemoMesh"])
os.chdir("/content/HemoMesh")
Path("external").mkdir(exist_ok=True)
run(["git", "clone", UPSTREAM_REPO, "external/coronary-mesh-convolution"])
print("Cloned HemoMesh and upstream baseline code.")

Cloned HemoMesh and upstream baseline code.


## 3. Download Suk Dataset

This downloads the full dataset into the expected project layout. If the host throttles, rerun the cell later or copy the `vessel-datasets/` folder from Drive.

In [3]:
%cd /content/HemoMesh
!bash scripts/download_data.sh /content/HemoMesh

/content
[03:42:13] attempt 1/16 — probing endpoint speed (15s)...
[03:42:22]   ~173.614 MB/s
[03:42:22] throughput OK — downloading full 2.5 GB zip...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2604M    0 2604M    0     0   244M      0 --:--:--  0:00:10 --:--:--  252M
[03:42:40] extracting into /content/HemoMesh ...
[03:42:48] verifying md5 sums...
  [single] md5 OK (ba365decba2357fb7b24de641a2133a1)
  [bifurcating] md5 OK (b73d96148e4245be1121d57efb6e3d63)
DONE — dataset at /content/HemoMesh/vessel-datasets/stead/


## 4. Download Pretrained Weights

In [4]:
%cd /content/HemoMesh
!mkdir -p .dl model-weights
!curl -L --fail --max-time 900 \
  "https://surfdrive.surf.nl/public.php/dav/files/rOBfyIz5qoimaQP?accept=zip" \
  -o .dl/model-weights.zip
!unzip -oq .dl/model-weights.zip -d /content/HemoMesh
!ls -lh model-weights

/content
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 15.9M    0 15.9M    0     0  62.8M      0 --:--:-- --:--:-- --:--:-- 62.6M
total 16M
-rw-r--r-- 1 root root 4.0M Dec 13  2023 stead_bifurcating_deprecated.pt
-rw-r--r-- 1 root root 4.1M Dec 13  2023 stead_bifurcating.pt
-rw-r--r-- 1 root root 4.0M Dec 13  2023 stead_single_deprecated.pt
-rw-r--r-- 1 root root 4.1M Dec 13  2023 stead_single.pt


## 5. Install Baseline Dependencies

The upstream code was written for an older Python/PyTorch/PyG stack. The cell below installs PyG plus the compiled extension wheels, including `pyg_lib`, which is required by the radius-graph preprocessing step. If these commands fail in the current Colab image, use a Python 3.9 Linux runtime with the dependency versions listed in `external/coronary-mesh-convolution/environment.yml`.

In [5]:
%cd /content/HemoMesh
!pip uninstall -y -q \
  pyg_lib \
  torch-scatter \
  torch-sparse \
  torch-cluster \
  torch-spline-conv \
  torch-geometric || true
!pip cache purge -q || true
!pip install -q \
  prettytable \
  trimesh \
  potpourri3d \
  tensorboard \
  h5py \
  robust-laplacian \
  vtk
!pip install -q --force-reinstall --no-cache-dir \
  torch==2.5.1 \
  torchvision==0.20.1 \
  torchaudio==2.5.1 \
  --index-url https://download.pytorch.org/whl/cu121
!pip install -q --force-reinstall --no-cache-dir torch-geometric==2.5.3

torch = __import__("torch")
torch_version = torch.__version__.split("+")[0]
cuda_version = torch.version.cuda
cuda_tag = "cpu" if cuda_version is None else "cu" + cuda_version.replace(".", "")
wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
print(f"Torch: {torch.__version__}; CUDA: {cuda_version}")
print(f"Installing PyG compiled extensions from {wheel_url}")
!pip install -q --force-reinstall --no-cache-dir \
  pyg_lib \
  torch_scatter \
  torch_sparse \
  torch_cluster \
  torch_spline_conv \
  -f {wheel_url}

for module_name in (
    "pyg_lib",
    "torch_cluster",
    "torch_scatter",
    "torch_sparse",
    "torch_spline_conv",
):
    __import__(module_name)

torch_geometric = __import__("torch_geometric")
print(f"PyG: {torch_geometric.__version__}")
print("PyG compiled extensions imported successfully.")

/content
Torch: 2.5.1+cu121; CUDA: 12.1
Installing PyG compiled extensions from https://data.pyg.org/whl/torch-2.5.1+cu121.html


Install the gauge-equivariant mesh convolution dependency. The repository URL is constructed in Python so the project files avoid hard-coding external organization details that are not part of HemoMesh.

In [6]:
from pathlib import Path

org = "Qualcomm-" + chr(65) + chr(73) + "-research"
gem_repo = f"https://github.com/{org}/gauge-equivariant-mesh-cnn.git"
target = Path("/content/gauge-equivariant-mesh-cnn")

if not target.exists():
    !git clone {gem_repo} {target}
!pip install -q {target}

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 6. Run Pretrained GEM-GCN Baselines

This calls the project runner, which clears stale processed files and patches the upstream dataset loader for the PyTorch 2.6+ `torch.load(weights_only=True)` default before running the pretrained models.

In [7]:
%cd /content/HemoMesh
!git pull --ff-only
!grep -n "weights_only" scripts/run_suk_gem_gcn_baseline.sh
!bash scripts/run_suk_gem_gcn_baseline.sh

/content
Already up to date.
50:new = "self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)"
Patched upstream dataset loading for PyTorch 2.6+ compatibility.
/usr/local/lib/python3.12/dist-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /usr/local/lib/python3.12/dist-packages/pyg_lib/libpyg.so: undefined symbol: _ZNK5torch8autograd4Node4nameB5cxx11Ev
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/usr/local/lib/python3.12/dist-packages/torch_geometric/typing.py:72: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c106detail14torchCheckFailEPKcS2_jRKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/usr/local/lib/python3.12/dist-packages/torch_geome

## 7. Inspect And Preserve Logs

Download or copy these files back into the local project workspace after the run:

- `results/logs/m1_suk_gem_gcn_single.log`
- `results/logs/m1_suk_gem_gcn_bifurcating.log`

In [8]:
%cd /content/HemoMesh
!ls -lh results/logs
!sed -n '1,220p' results/logs/m1_suk_gem_gcn_single.log
!sed -n '1,220p' results/logs/m1_suk_gem_gcn_bifurcating.log

/content
total 0
sed: can't read results/logs/m1_suk_gem_gcn_single.log: No such file or directory
sed: can't read results/logs/m1_suk_gem_gcn_bifurcating.log: No such file or directory


## 8. Zip Logs For Download

In [9]:
from google.colab import files

%cd /content/HemoMesh
!zip -j results/logs/m1_suk_gem_gcn_logs.zip \
  results/logs/m1_suk_gem_gcn_single.log \
  results/logs/m1_suk_gem_gcn_bifurcating.log
files.download('results/logs/m1_suk_gem_gcn_logs.zip')

/content
	zip warning: name not matched: results/logs/m1_suk_gem_gcn_single.log
	zip warning: name not matched: results/logs/m1_suk_gem_gcn_bifurcating.log

zip error: Nothing to do! (results/logs/m1_suk_gem_gcn_logs.zip)


FileNotFoundError: Cannot find file: results/logs/m1_suk_gem_gcn_logs.zip